# PINN 1 — 1D steady heat equation

A **physics-informed neural network (PINN)** is trained to satisfy a differential equation
rather than (or in addition to) labelled data: the PDE residual, evaluated with automatic
differentiation, is part of the loss.

**Problem**

$$\frac{d^2u}{dx^2} = -\sin(\pi x), \qquad x\in[0,1], \qquad u(0)=u(1)=0,$$

with exact solution $u(x)=\sin(\pi x)/\pi^2$ — the simplest possible test case (1D, linear,
steady) and a good way to see every ingredient of a PINN.

| | |
|---|---|
| Problem class | `neural_pde_solvers.pinn.Heat1D` |
| Config | `configs/pinn/heat1d.yaml` |
| CLI | `pinn-run configs/pinn/heat1d.yaml` |

In [ ]:
# --- Setup -------------------------------------------------------------------
# Locally: `pip install -e ".[pinn]"` from the repository root.
# On Google Colab the package is installed from GitHub automatically.
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("neural_pde_solvers") is None:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "neural-pde-solvers[pinn] @ git+https://github.com/tvh0ng/neural-pde-solvers.git",
        ]
    )

import matplotlib.pyplot as plt
import numpy as np
import torch

from neural_pde_solvers import set_seed, setup_logging
from neural_pde_solvers.pinn import (
    MLP,
    PINNTrainer,
    TrainerConfig,
    count_parameters,
    evaluate,
    grid,
    predict,
    resolve_device,
)
from neural_pde_solvers.pinn.plotting import plot_loss_history

setup_logging()
set_seed(42)
device = resolve_device("auto")
QUICK = os.environ.get("PINN_QUICK") == "1"  # shortened run used by CI
print(f"PyTorch {torch.__version__} | device: {device}")

## 1. Problem: residual and boundary loss

`Heat1D` implements the residual with autograd (see `src/neural_pde_solvers/pinn/problems.py`):

```python
def residual(self, model, X):
    u = model(X)
    return laplacian(u, X) - self.source(X)      # u'' + sin(pi x)
```

and the loss terms

$$\mathcal L = \underbrace{\tfrac1{N}\textstyle\sum_i \mathcal R(x_i)^2}_{\text{pde}}
+ w_{bc}\,\underbrace{\tfrac12\big(u(0)^2 + u(1)^2\big)}_{\text{bc}}, \qquad w_{bc}=10.$$

No measured data is needed: the equation itself supervises the network at randomly drawn
**collocation points**, which are re-sampled every epoch.

In [ ]:
from neural_pde_solvers.pinn import Heat1D

problem = Heat1D().to(device)
print("domain:", problem.domain(), "| default weights:", problem.default_weights)

## 2. Network

A small `tanh` MLP. Smooth activations matter: the loss contains $u''$, and ReLU has a
second derivative that is zero almost everywhere.

In [ ]:
model = MLP(in_dim=1, out_dim=1, width=32, depth=3, activation="tanh").to(device)
print(model)
print(f"trainable parameters: {count_parameters(model):,}")

## 3. Training
Adam with `ReduceLROnPlateau`; 100 collocation points re-drawn each epoch.

In [ ]:
config = TrainerConfig(
    epochs=500 if QUICK else 5000,
    lr=1e-3,
    n_collocation=100,
    scheduler="plateau",
    scheduler_kwargs=dict(factor=0.5, patience=500, min_lr=1e-6),
    log_every=500,
)
trainer = PINNTrainer(model, problem, config, weights={"pde": 1.0, "bc": 10.0})
history = trainer.fit()

## 4. Results

In [ ]:
mesh, X = grid(problem, 201)
x = X[:, 0]
u_pred, u_exact = predict(model, X)[:, 0], problem.exact(X)[:, 0]
metrics = evaluate(model, problem, 201)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))
axes[0].plot(x, u_exact, lw=2, label="exact")
axes[0].plot(x, u_pred, "--", lw=2, label="PINN")
axes[0].scatter([0, 1], [0, 0], c="k", zorder=5, label="boundary conditions")
axes[0].set(title="Solution", xlabel="x", ylabel="u(x)")
axes[0].legend()
axes[1].semilogy(x, np.abs(u_pred - u_exact))
axes[1].set(title=f"|error|  (relative L2 = {metrics['rel_l2']:.2e})", xlabel="x")
plot_loss_history(history, ax=axes[2])
for ax in axes[:2]:
    ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
metrics

## Key takeaways

1. The PINN learns the solution from the equation and boundary conditions alone.
2. Automatic differentiation provides $u'$ and $u''$ exactly (no finite differences).
3. Practical defaults: `tanh` activations, Xavier initialisation, re-sampled collocation
   points, a larger weight on boundary terms, and learning-rate scheduling.

**Try:** fewer/more collocation points, a different source term, or
`TrainerConfig(lbfgs_iters=500)` for an L-BFGS refinement stage.
Next: **PINN 2 — Burgers' equation** (time-dependent, nonlinear).